# Week 09 과제 · 모범답안 (교수자용)

> 교수자용 모범답안입니다. 학생에게 배포하지 마세요.

## 준비: 데이터와 자동 확인 도우미

In [1]:
import numpy as np
from pathlib import Path
import urllib.request

DATA_URL = "https://raw.githubusercontent.com/MNU-AI-Programming/AI-Assisted-Python-Programming/main/datasets/"

def get_data(name):
    for folder in [Path("../../datasets"), Path("datasets"), Path(".")]:
        if (folder / name).exists():
            return str(folder / name)
    urllib.request.urlretrieve(DATA_URL + name, name)
    return name

WEATHER = get_data("weather_daily.csv")
AVG_T = np.genfromtxt(WEATHER, delimiter=",", skip_header=1, usecols=1)   # 평균기온 (결측 3개 포함)


def check(desc, test):
    """자동 확인 도우미: test는 True/False를 돌려주는 함수(lambda)입니다."""
    try:
        ok = test()
    except NotImplementedError:
        print(f"⏳ 미작성 - {desc}")
    except Exception as e:
        print(f"❌ 오류 - {desc} ({type(e).__name__}: {e})")
    else:
        print(f"{'✅ 통과' if ok else '❌ 실패'} - {desc}")


def same(a, b):
    """배열·숫자를 비교 (NaN 위치도 같아야 함)."""
    a, b = np.asarray(a, dtype=float), np.asarray(b, dtype=float)
    return a.shape == b.shape and np.allclose(a, b, equal_nan=True)

---
## 문제 1. 점수 행렬 요약 (2점)

`scores`는 shape `(학생 수, 과목 수)`인 배열입니다. **결측(`np.nan`)이 있을 수 있으며 무시합니다.**

1. `student_avg(scores)`: 학생별 평균, 소수 첫째 자리 반올림 → shape `(학생 수,)`
2. `subject_best(scores)`: 과목별 최고점을 받은 **학생 번호** → shape `(과목 수,)`
3. `count_all_pass(scores, cut=60)`: **모든 과목이** `cut` 이상인 학생 수 (결측 과목은 통과로 보지 않음)

In [2]:
def student_avg(scores):
    return np.round(np.nanmean(scores, axis=1), 1)


def subject_best(scores):
    return np.nanargmax(scores, axis=0)


def count_all_pass(scores, cut=60):
    return int((scores >= cut).all(axis=1).sum())

In [3]:
S = np.array([[78, 85, 90],
              [92, np.nan, 95],
              [65, 70, 59],
              [88, 91, 84]], dtype=float)
check("student_avg", lambda: same(student_avg(S), [84.3, 93.5, 64.7, 87.7]))
check("student_avg shape", lambda: student_avg(S).shape == (4,))
check("subject_best", lambda: list(subject_best(S)) == [1, 3, 1])
check("count_all_pass (결측은 통과 아님)", lambda: count_all_pass(S) == 2)
check("count_all_pass cut=80", lambda: count_all_pass(S, 80) == 1)

✅ 통과 - student_avg
✅ 통과 - student_avg shape
✅ 통과 - subject_best
✅ 통과 - count_all_pass (결측은 통과 아님)
✅ 통과 - count_all_pass cut=80


---
## 문제 2. 정규화 두 가지 (2점)

서로 단위가 다른 값을 비교할 때 값을 같은 범위로 맞춥니다.

1. `minmax(x)`: `(x − 최솟값) / (최댓값 − 최솟값)` → 0~1. **모든 값이 같으면 0으로 채운 배열**을 돌려줍니다.
2. `zscore(x)`: `(x − 평균) / 표준편차` (모집단 표준편차, `np.std` 기본값). 표준편차가 0이면 0으로 채운 배열.

두 함수 모두 NaN은 계산에서 무시하고, **결과에서도 NaN 위치는 NaN**으로 둡니다. (`np.nanmin`, `np.nanmean`, `np.nanstd` 활용)

In [4]:
def minmax(x):
    x = np.asarray(x, dtype=float)
    lo, hi = np.nanmin(x), np.nanmax(x)
    if hi == lo:
        return np.where(np.isnan(x), np.nan, 0.0)
    return (x - lo) / (hi - lo)


def zscore(x):
    x = np.asarray(x, dtype=float)
    sd = np.nanstd(x)
    if sd == 0:
        return np.where(np.isnan(x), np.nan, 0.0)
    return (x - np.nanmean(x)) / sd

In [5]:
check("minmax 기본", lambda: same(minmax([10, 20, 30]), [0, 0.5, 1]))
check("minmax NaN 유지", lambda: same(minmax([10, np.nan, 30]), [0, np.nan, 1]))
check("minmax 모두 같은 값", lambda: same(minmax([5, 5, 5]), [0, 0, 0]))
check("zscore 기본", lambda: same(zscore([2, 4, 6]), [-1.224744871, 0, 1.224744871]))
check("zscore 평균 0, 표준편차 1", lambda: abs(np.nanmean(zscore(AVG_T))) < 1e-9 and abs(np.nanstd(zscore(AVG_T)) - 1) < 1e-9)
check("zscore 표준편차 0", lambda: same(zscore([3, 3]), [0, 0]))
check("입력 리스트도 처리", lambda: isinstance(minmax([1, 2]), np.ndarray))

✅ 통과 - minmax 기본
✅ 통과 - minmax NaN 유지
✅ 통과 - minmax 모두 같은 값
✅ 통과 - zscore 기본
✅ 통과 - zscore 평균 0, 표준편차 1
✅ 통과 - zscore 표준편차 0
✅ 통과 - 입력 리스트도 처리


---
## 문제 3. IQR 이상값 탐지 (3점)

**IQR(사분위 범위) 방법**: Q1(25%), Q3(75%), IQR = Q3 − Q1 일 때
`값 < Q1 − 1.5×IQR` 또는 `값 > Q3 + 1.5×IQR` 이면 이상값입니다.

1. `iqr_outliers(x, k=1.5)`: 이상값이면 `True`인 **불리언 배열**. NaN은 `False`. 사분위수는 NaN을 무시하고 구합니다 (`np.nanpercentile`).
2. `clean_mean(x)`: 이상값과 NaN을 **뺀** 평균 (소수 둘째 자리 반올림)

In [6]:
def iqr_outliers(x, k=1.5):
    x = np.asarray(x, dtype=float)
    q1, q3 = np.nanpercentile(x, [25, 75])
    iqr = q3 - q1
    with np.errstate(invalid="ignore"):
        return (x < q1 - k * iqr) | (x > q3 + k * iqr)


def clean_mean(x):
    x = np.asarray(x, dtype=float)
    keep = ~iqr_outliers(x) & ~np.isnan(x)
    return round(float(x[keep].mean()), 2)

In [7]:
v = np.array([10, 12, 11, 13, 12, 95, 11, np.nan, -40, 12], dtype=float)
check("이상값 위치", lambda: list(np.where(iqr_outliers(v))[0]) == [5, 8])
check("NaN은 False", lambda: bool(iqr_outliers(v)[7]) is False)
check("반환은 bool 배열", lambda: iqr_outliers(v).dtype == bool and iqr_outliers(v).shape == v.shape)
check("k를 키우면 덜 엄격", lambda: iqr_outliers(v, k=100).sum() == 0)
check("clean_mean", lambda: clean_mean(v) == 11.57)
check("기온 데이터: 이상값 없음 (계절 변화는 정상)", lambda: iqr_outliers(AVG_T).sum() == 0)

✅ 통과 - 이상값 위치
✅ 통과 - NaN은 False
✅ 통과 - 반환은 bool 배열
✅ 통과 - k를 키우면 덜 엄격
✅ 통과 - clean_mean
✅ 통과 - 기온 데이터: 이상값 없음 (계절 변화는 정상)


---
## 문제 4. 이동평균과 연속 상승 (AI 협업, 3점)

1. `moving_average(x, n)`: 길이 `len(x) − n + 1`인 n일 이동평균. `np.convolve`를 사용하고, `n`이 1보다 작거나 `len(x)`보다 크면 `ValueError`.
2. `longest_rise(x)`: 값이 **전날보다 엄격하게 커진 날**이 가장 길게 이어진 일수. (예: `[1, 2, 3, 2, 3]` → 2→3 두 번 연속이 처음 구간 → `2`). NaN이 있으면 그 지점에서 연속이 끊깁니다. 원소가 0~1개면 `0`.
   - 이 함수는 반복문을 써도 됩니다. 도전: `np.diff`를 활용해 보세요.

**AI 협업 과제**: AI에게 `longest_rise`를 요청할 때 **"연속 상승 일수"의 정의**(상승한 날의 수인지, 구간의 날짜 수인지)를 명확히 했는지 기록하세요. AI 답이 정의와 다르게 계산하는지 작은 예제로 확인합니다.

In [8]:
def moving_average(x, n):
    x = np.asarray(x, dtype=float)
    if n < 1 or n > len(x):
        raise ValueError(f"창 크기 오류: {n}")
    return np.convolve(x, np.ones(n) / n, mode="valid")


def longest_rise(x):
    x = np.asarray(x, dtype=float)
    if len(x) < 2:
        return 0
    with np.errstate(invalid="ignore"):
        up = np.diff(x) > 0          # NaN과의 비교는 False → 연속이 끊긴다
    best = cur = 0
    for flag in up:
        cur = cur + 1 if flag else 0
        best = max(best, cur)
    return best

In [9]:
def _raises_value_error(fn, *args):
    try:
        fn(*args)
    except ValueError:
        return True
    return False

check("이동평균 값", lambda: same(moving_average([1, 2, 3, 4, 5], 3), [2, 3, 4]))
check("이동평균 길이", lambda: len(moving_average(np.arange(365), 7)) == 359)
check("n=1이면 그대로", lambda: same(moving_average([3, 1, 2], 1), [3, 1, 2]))
check("잘못된 n → ValueError", lambda: _raises_value_error(moving_average, [1, 2], 3) and _raises_value_error(moving_average, [1, 2], 0))
check("longest_rise 기본", lambda: longest_rise([1, 2, 3, 2, 3]) == 2)
check("같은 값은 상승 아님", lambda: longest_rise([1, 1, 1]) == 0)
check("마지막 구간이 가장 김", lambda: longest_rise([5, 4, 1, 2, 3, 4]) == 3)
check("NaN에서 끊김", lambda: longest_rise([1, 2, np.nan, 3, 4, 5]) == 2)
check("원소 0~1개", lambda: longest_rise([]) == 0 and longest_rise([7]) == 0)
check("기온 데이터 최장 연속 상승", lambda: longest_rise(AVG_T) == 4)

✅ 통과 - 이동평균 값
✅ 통과 - 이동평균 길이
✅ 통과 - n=1이면 그대로
✅ 통과 - 잘못된 n → ValueError
✅ 통과 - longest_rise 기본
✅ 통과 - 같은 값은 상승 아님
✅ 통과 - 마지막 구간이 가장 김
✅ 통과 - NaN에서 끊김
✅ 통과 - 원소 0~1개
✅ 통과 - 기온 데이터 최장 연속 상승


### AI 활용 기록 (예시)

| 항목 | 내용 |
|---|---|
| 사용한 AI 도구 | (예) Gemini |
| 정의 | "전날보다 엄격하게 큰 날의 연속 개수. [1,2,3,2,3] → 2" 처럼 예시를 함께 줌 |
| AI 답의 차이 | 처음에는 구간의 날짜 수(3)를 셌고, `>=`를 써서 같은 값도 상승으로 셌으며, NaN 처리가 없었음 |
| 직접 고친 부분 | `np.diff(x) > 0`으로 상승 여부를 만들고, NaN 비교가 False가 되는 성질을 이용해 연속을 끊음 |

---
## 성찰 (필수)

1. 반복문 대신 배열 연산으로 바꾸면서 생각하는 방식이 어떻게 달라졌나요?
2. `axis`나 `shape` 때문에 헷갈린 경험과, 그것을 확인한 방법은?
3. 정의가 애매한 요구사항(예: 연속 상승 일수)을 AI에게 줄 때 무엇을 함께 주면 좋을까요?

(학생 작성 영역)